<a href="https://colab.research.google.com/github/DivyaJaviya01/minecraft-server/blob/main/Minecraft-1.20-Colab-v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minecraft 1.20.x Colab Server - Paper + Fabric
Upload this file to Google Colab. Run cells top to bottom.

**Xaero Map = install on your PC only.** No server file needed. Works on Paper AND Fabric.
Get Xaero Minimap + World Map from CurseForge / Modrinth, put in your PC `mods` folder with Fabric/Forge client.

In [1]:
# CELL 1 - Drive + Java 17 (best for 1.20.x)
from google.colab import drive
from pathlib import Path
import subprocess, shutil

drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/MinecraftServers')
ROOT.mkdir(parents=True, exist_ok=True)
print('Root:', ROOT)

if shutil.which('java') is None:
    print('Installing Java 17...')
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','-qq','openjdk-17-jre-headless'], check=True)
else:
    print('Java found:', shutil.which('java'))

subprocess.run(['java','-version'])
JAVA = shutil.which('java')
print('JAVA =', JAVA)


Mounted at /content/drive
Root: /content/drive/MyDrive/MinecraftServers
Java found: /usr/bin/java
JAVA = /usr/bin/java


In [2]:
# CELL 2 - Choose your server (edit these 4 lines, then Run)
# @markdown --- Fill this, then Run ---
SERVER_NAME = 'my-smp' # @param {type:"string"}
LOADER = 'Fabric' # @param ["Paper", "Fabric"]
MC_VERSION = '1.20.1' # @param ["1.20.1", "1.20.4"]
RAM_GB = 4 # @param {type:"slider", min:2, max:10, step:1}

import json
SERVER_DIR = ROOT / SERVER_NAME
SERVER_FILES = SERVER_DIR / 'server'          # real server lives here
MODS_DIR = SERVER_FILES / 'mods'              # FIXED: mods INSIDE server folder
PLUGINS_DIR = SERVER_FILES / 'plugins'        # plugins INSIDE server folder
BACKUPS_DIR = SERVER_DIR / 'backups'
for d in [SERVER_DIR, SERVER_FILES, MODS_DIR, PLUGINS_DIR, BACKUPS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

config = {'server_name': SERVER_NAME, 'loader': LOADER, 'minecraft_version': MC_VERSION, 'ram': RAM_GB, 'java': 17}
(SERVER_DIR/'config.json').write_text(json.dumps(config, indent=2))
print(json.dumps(config, indent=2))
print('Server folder:', SERVER_FILES)
print('Put extra .jar mods/plugins in:', MODS_DIR if LOADER=='Fabric' else PLUGINS_DIR)


{
  "server_name": "my-smp",
  "loader": "Fabric",
  "minecraft_version": "1.20.1",
  "ram": 4,
  "java": 17
}
Server folder: /content/drive/MyDrive/MinecraftServers/my-smp/server
Put extra .jar mods/plugins in: /content/drive/MyDrive/MinecraftServers/my-smp/server/mods


In [3]:
# CELL 3 - Download Paper / Fabric (tested)
import requests, os, shutil, subprocess

def dl(url, dest):
    print(f'Downloading -> {dest}')
    with requests.get(url, stream=True, headers={'User-Agent':'colab-mc/2.0'}, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        done = 0
        with open(dest,'wb') as f:
            for c in r.iter_content(1024*64):
                if c:
                    f.write(c); done += len(c)
                    if total: print(f'\r{done*100/total:.1f}%', end='')
    print('\nDone:', os.path.getsize(dest)/1024/1024, 'MB')

ver = config['minecraft_version']
if config['loader']=='Paper':
    JAR = str(SERVER_FILES/'server.jar')
    meta = requests.get(f'https://fill.papermc.io/v3/projects/paper/versions/{ver}/builds/latest', headers={'User-Agent':'colab-mc/2.0'}, timeout=30).json()
    url = meta['downloads']['server:default']['url']
    print('Paper build:', meta['id'], '| channel:', meta.get('channel'))
    dl(url, JAR)
else:
    lm = requests.get(f'https://meta.fabricmc.net/v2/versions/loader/{ver}', timeout=30).json()[0]
    loader_ver = lm['loader']['version']
    print('Fabric loader:', loader_ver)
    inst = str(SERVER_DIR/'fabric-installer.jar')
    dl('https://maven.fabricmc.net/net/fabricmc/fabric-installer/1.0.1/fabric-installer-1.0.1.jar', inst)
    subprocess.run([JAVA,'-jar',inst,'server','-mcversion',ver,'-loader',loader_ver,'-downloadMinecraft','-dir',str(SERVER_FILES)], check=True)
    JAR = str(SERVER_FILES/'fabric-server-launch.jar')
    print('Fabric ready. MAIN JAR =', JAR)
    print('Do NOT use server.jar or versions/server-1.20.1.jar (vanilla, ignores mods).')
print('MAIN JAR:', JAR)


Fabric loader: 0.19.5
100.0%
Done: 0.18870925903320312 MB
Fabric ready. MAIN JAR = /content/drive/MyDrive/MinecraftServers/my-smp/server/fabric-server-launch.jar
Do NOT use server.jar or versions/server-1.20.1.jar (vanilla, ignores mods).
MAIN JAR: /content/drive/MyDrive/MinecraftServers/my-smp/server/fabric-server-launch.jar


In [4]:
# CELL 4 - One-click packs: map + claims + essentials (tested Modrinth slugs)
# No voice chat (you use Discord) - skipped on purpose
INSTALL_PERF = True # @param {type:"boolean"}
INSTALL_MAP = True # @param {type:"boolean"}
INSTALL_CLAIMS_HOME = True # @param {type:"boolean"}
import requests, os

def modrinth(slug, loaders, game_ver, dest_dir):
    try:
        q = f'https://api.modrinth.com/v2/project/{slug}/version?game_versions=%5B%22{game_ver}%22%5D&loaders=%5B{loaders}%5D&limit=1'
        v = requests.get(q, headers={'User-Agent':'colab-mc/2.0'}, timeout=30).json()[0]
        f = v['files'][0]
        url, fname = f['url'], f['filename']
        out = os.path.join(dest_dir, fname)
        if os.path.exists(out): print(f'Skip {fname} (exists)'); return True
        print(f'DL {slug} -> {fname}')
        with requests.get(url, stream=True, timeout=120) as r:
            r.raise_for_status()
            open(out,'wb').write(r.content)
        return True
    except Exception as e:
        print(f'FAIL {slug}: {e}'); return False

ver = config['minecraft_version']
if config['loader']=='Fabric':
    dest = str(MODS_DIR)
    L='"fabric"'
    if INSTALL_PERF:
        for m in ['fabric-api','lithium','ferrite-core','krypton','cloth-config','spark']: modrinth(m, L, ver, dest)
    if INSTALL_MAP:
        modrinth('bluemap', L, ver, dest)
        print('NOTE: Xaero Minimap/WorldMap install on YOUR PC, not server. It will just work.')
    if INSTALL_CLAIMS_HOME:
        modrinth('flan', L, ver, dest)
        print('Claims: /flan claim. Homes: vanilla /spawnpoint + bed, or add Essential Commands mod manually.')
else:
    dest = str(PLUGINS_DIR)
    L='"paper"%2C"bukkit"%2C"purpur"'
    if INSTALL_PERF: print('Skip spark on Paper (not on Modrinth). Get it from Hangar manually if needed.')
    if INSTALL_MAP: modrinth('squaremap', L, ver, dest)
    if INSTALL_CLAIMS_HOME:
        for m in ['luckperms','essentialsx','griefprevention']: modrinth(m, L, ver, dest)
        print('Essentials: /sethome /home /tpa | Claims: golden shovel.')
print('Done. Files in:', dest)


Skip fabric-api-0.92.12+1.20.1.jar (exists)
Skip lithium-fabric-mc1.20.1-0.11.4.jar (exists)
Skip ferritecore-6.0.1-fabric.jar (exists)
Skip krypton-0.2.3.jar (exists)
Skip cloth-config-11.1.136-fabric.jar (exists)
Skip spark-1.10.53-fabric.jar (exists)
Skip BlueMap-5.3-fabric-1.20.jar (exists)
NOTE: Xaero Minimap/WorldMap install on YOUR PC, not server. It will just work.
Skip flan-1.20.1-1.11.16-fabric.jar (exists)
Claims: /flan claim. Homes: vanilla /spawnpoint + bed, or add Essential Commands mod manually.
Done. Files in: /content/drive/MyDrive/MinecraftServers/my-smp/server/mods


In [ ]:
# CELL 5 - EULA + first run (Fabric uses fabric-server-launch.jar)
import subprocess, time, sys, os, re
(SERVER_FILES/'eula.txt').write_text('eula=true\n')
prop = SERVER_FILES/'server.properties'
OFFLINE_MODE = True # @param {type:"boolean"}  # True for TLauncher/cracked, False for paid accounts
if config['loader']=='Fabric':
    JAR = str(SERVER_FILES/'fabric-server-launch.jar')
else:
    JAR = str(SERVER_FILES/'server.jar')
print('Using jar:', JAR, os.path.getsize(JAR), 'bytes')
p = subprocess.Popen([JAVA, f"-Xms{config['ram']}G", f"-Xmx{config['ram']}G", '-jar', JAR, '--nogui'], cwd=str(SERVER_FILES), stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
start=time.time(); ok=False; last_heartbeat=start
while time.time()-start<600:
    line=p.stdout.readline()
    if not line:
        if p.poll() is not None: print('Process exited early, code', p.poll()); break
        continue
    print(line.rstrip())
    if 'Done (' in line or 'For help, type' in line: ok=True; break
    if 'Failed to load properties' in line: print('>> first-run notice: server.properties not yet created, normal. Continuing...')
    if time.time()-last_heartbeat>30: print(f">> still working... {int(time.time()-start)}s (first world gen is slow on Colab, wait for Done)" ); last_heartbeat=time.time()
if ok:
    print('First run OK, stopping...')
    try:
        p.stdin.write('stop\n'); p.stdin.flush()
    except Exception as e:
        print('stdin stop failed, killing:', e); p.kill()
    try: p.wait(timeout=90)
    except Exception: p.kill()
    print('Stopped.')
else:
    print('Did not see Done yet. Server may still be generating spawn (0% loop is normal first time).')
    print('Leave it 2-3 min, or stop with: p.stdin.write("stop\\n")')
if prop.exists():
    t=prop.read_text()
    for k,v in [('view-distance','6'),('simulation-distance','4'),('max-tick-time','-1')]:
        t=re.sub(f'^{k}=.*',f'{k}={v}',t,flags=re.M) if f'{k}=' in t else t+f'\n{k}={v}\n'
    prop.write_text(t); print('Patched server.properties for low lag')
    om = 'false' if OFFLINE_MODE else 'true'
    t2 = prop.read_text()
    import re as _re
    t2 = _re.sub('^online-mode=.*', f'online-mode={om}', t2, flags=_re.M) if 'online-mode=' in t2 else t2 + f'\nonline-mode={om}\n'
    prop.write_text(t2); print(f'online-mode={om} (OFFLINE_MODE={OFFLINE_MODE})')


In [ ]:
# AUTO-RECOVER (if you restarted Colab, run Cell 1+2 first. This also tries auto-load)
try:
    config; SERVER_FILES; JAVA
except NameError:
    from pathlib import Path
    import json as _json, shutil as _shutil
    try:
        from google.colab import drive as _drive
        _drive.mount('/content/drive')
    except Exception: pass
    _ROOT = Path('/content/drive/MyDrive/MinecraftServers')
    _cands = [f for f in _ROOT.iterdir() if f.is_dir() and (f/'config.json').exists()] if _ROOT.exists() else []
    if not _cands: raise RuntimeError('No server found. Run Cell 1+2 first with same SERVER_NAME.')
    _pick = [f for f in _cands if f.name=='my-smp']
    SERVER_DIR = (_pick[0] if _pick else _cands[0])
    config = _json.loads((SERVER_DIR/'config.json').read_text())
    SERVER_NAME = config['server_name']
    SERVER_FILES = SERVER_DIR/'server'
    MODS_DIR = SERVER_FILES/'mods'
    PLUGINS_DIR = SERVER_FILES/'plugins'
    BACKUPS_DIR = SERVER_DIR/'backups'
    ROOT = _ROOT
    JAVA = _shutil.which('java')
    print(f"Recovered: {SERVER_NAME} | {config['loader']} {config['minecraft_version']}")

# CELL 6 - Start server (keep running) + commands
import subprocess, threading, sys, os
if config['loader']=='Fabric':
    JAR = str(SERVER_FILES/'fabric-server-launch.jar')
else:
    JAR = str(SERVER_FILES/'server.jar')
print('Starting with:', JAR)
SERVER_PROCESS = subprocess.Popen([JAVA, f"-Xms{config['ram']}G", f"-Xmx{config['ram']}G", '-jar', JAR, '--nogui'], cwd=str(SERVER_FILES), stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
def _log():
    for line in SERVER_PROCESS.stdout:
        print(line.rstrip()); sys.stdout.flush()
threading.Thread(target=_log, daemon=True).start()
print('Server starting... wait for Done (...) then do the TUNNEL cells below.')

def send(c):
    SERVER_PROCESS.stdin.write(c+'\n'); SERVER_PROCESS.stdin.flush(); print('[CMD]',c)
def stop(): send('stop')
def save(): send('save-all')
# usage: send('say hello')  send('list')  send('op YourName')  stop()


In [ ]:
# AUTO-LOAD (if kernel restarted, finds your server in Drive)
try:
    config; SERVER_FILES; JAVA; SERVER_DIR
except NameError:
    from pathlib import Path
    import json as _json, shutil as _shutil
    try:
        from google.colab import drive as _drive
        _drive.mount('/content/drive')
    except Exception: pass
    _ROOT = Path('/content/drive/MyDrive/MinecraftServers')
    _cands = [f for f in _ROOT.iterdir() if f.is_dir() and (f/'config.json').exists()] if _ROOT.exists() else []
    if not _cands: raise RuntimeError('No server found. Run Cell 1+2 first with same SERVER_NAME.')
    _pick = [f for f in _cands if f.name=='my-smp']
    SERVER_DIR = (_pick[0] if _pick else _cands[0])
    config = _json.loads((SERVER_DIR/'config.json').read_text())
    SERVER_NAME = config['server_name']
    SERVER_FILES = SERVER_DIR/'server'
    MODS_DIR = SERVER_FILES/'mods'
    PLUGINS_DIR = SERVER_FILES/'plugins'
    BACKUPS_DIR = SERVER_DIR/'backups'
    ROOT = _ROOT
    JAVA = _shutil.which('java')
    print(f"Recovered: {SERVER_NAME} | {config['loader']} {config['minecraft_version']}")

# PROGRESS - shows % while world generates (run in separate cell, wait for DONE)
import time, re
from pathlib import Path
_log = SERVER_FILES/'logs'/'latest.log'
_start = time.time(); _seen = ''
while time.time() - _start < 900:
    _t = _log.read_text() if _log.exists() else ''
    _m = re.findall(r'Preparing spawn area: (\d+)%', _t)
    _pct = _m[-1] if _m else '0'
    if 'Done (' in _t:
        print('DONE - server ready!'); break
    if _pct != _seen:
        print(f'{_pct}% done - {int(time.time()-_start)}s passed'); _seen = _pct
    time.sleep(5)


In [ ]:
# AUTO-LOAD (if kernel restarted, finds your server in Drive)
try:
    config; SERVER_FILES; JAVA; SERVER_DIR
except NameError:
    from pathlib import Path
    import json as _json, shutil as _shutil
    try:
        from google.colab import drive as _drive
        _drive.mount('/content/drive')
    except Exception: pass
    _ROOT = Path('/content/drive/MyDrive/MinecraftServers')
    _cands = [f for f in _ROOT.iterdir() if f.is_dir() and (f/'config.json').exists()] if _ROOT.exists() else []
    if not _cands: raise RuntimeError('No server found. Run Cell 1+2 first with same SERVER_NAME.')
    _pick = [f for f in _cands if f.name=='my-smp']
    SERVER_DIR = (_pick[0] if _pick else _cands[0])
    config = _json.loads((SERVER_DIR/'config.json').read_text())
    SERVER_NAME = config['server_name']
    SERVER_FILES = SERVER_DIR/'server'
    MODS_DIR = SERVER_FILES/'mods'
    PLUGINS_DIR = SERVER_FILES/'plugins'
    BACKUPS_DIR = SERVER_DIR/'backups'
    ROOT = _ROOT
    JAVA = _shutil.which('java')
    print(f"Recovered: {SERVER_NAME} | {config['loader']} {config['minecraft_version']}")

# TUNNEL 7A - playit setup: run once, open claim URL, approve, then run 7B
import subprocess, os, stat, time
PLAY_DIR = SERVER_DIR/'playit'; PLAY_DIR.mkdir(exist_ok=True)
BIN = PLAY_DIR/'playit'; CLI = PLAY_DIR/'playit-cli'
import requests
for _name, _f in [('playit', BIN), ('playit-cli', CLI)]:
    if not _f.exists():
        print(f'Downloading {_name}...')
        _asset = 'playit-linux-amd64' if _name=='playit' else 'playit-cli-linux-amd64'
        open(_f,'wb').write(requests.get(f'https://github.com/playit-cloud/playit-agent/releases/download/v1.0.10/{_asset}', timeout=120).content)
        os.chmod(_f, os.stat(_f).st_mode | stat.S_IEXEC)
!pkill -9 -f /tmp/playit/playit; sleep 1
!mkdir -p /tmp/playit
!cp "$BIN" /tmp/playit/playit
!cp "$CLI" /tmp/playit/playit-cli
!chmod +x /tmp/playit/playit /tmp/playit/playit-cli
D = subprocess.Popen(['/tmp/playit/playit','--socket-path=/tmp/playit/playit.sock','--secret-path=/tmp/playit/playit.toml'], cwd='/tmp/playit', stdout=open('/tmp/d.log','w'), stderr=subprocess.STDOUT, text=True)
time.sleep(5)
print(open('/tmp/d.log').read()[-600:])
G = subprocess.run(['/tmp/playit/playit-cli','--socket-path=/tmp/playit/playit.sock','claim','generate'], capture_output=True, text=True, timeout=30)
CLAIM_CODE = G.stdout.strip().split()[-1]
U = subprocess.run(['/tmp/playit/playit-cli','--socket-path=/tmp/playit/playit.sock','claim','url',CLAIM_CODE], capture_output=True, text=True, timeout=30)
print('OPEN THIS URL, login, approve:')
print(U.stdout.strip())
print('Then run TUNNEL 7B.')


In [ ]:
# AUTO-LOAD (if kernel restarted, finds your server in Drive)
try:
    config; SERVER_FILES; JAVA; SERVER_DIR
except NameError:
    from pathlib import Path
    import json as _json, shutil as _shutil
    try:
        from google.colab import drive as _drive
        _drive.mount('/content/drive')
    except Exception: pass
    _ROOT = Path('/content/drive/MyDrive/MinecraftServers')
    _cands = [f for f in _ROOT.iterdir() if f.is_dir() and (f/'config.json').exists()] if _ROOT.exists() else []
    if not _cands: raise RuntimeError('No server found. Run Cell 1+2 first with same SERVER_NAME.')
    _pick = [f for f in _cands if f.name=='my-smp']
    SERVER_DIR = (_pick[0] if _pick else _cands[0])
    config = _json.loads((SERVER_DIR/'config.json').read_text())
    SERVER_NAME = config['server_name']
    SERVER_FILES = SERVER_DIR/'server'
    MODS_DIR = SERVER_FILES/'mods'
    PLUGINS_DIR = SERVER_FILES/'plugins'
    BACKUPS_DIR = SERVER_DIR/'backups'
    ROOT = _ROOT
    JAVA = _shutil.which('java')
    print(f"Recovered: {SERVER_NAME} | {config['loader']} {config['minecraft_version']}")

# TUNNEL 7B - run AFTER approving claim URL in browser
import subprocess, time
code = CLAIM_CODE if 'CLAIM_CODE' in globals() else input('Paste claim code: ').strip()
E = subprocess.run(['/tmp/playit/playit-cli','--socket-path=/tmp/playit/playit.sock','claim','exchange',code], capture_output=True, text=True, timeout=60)
print((E.stdout+E.stderr)[-800:])
time.sleep(8)
print(open('/tmp/d.log').read()[-800:])
print('Refresh playit.gg Agents page: agent online -> create TCP tunnel port 25565 -> join address shows.')


In [8]:
# AUTO-RECOVER (if you restarted Colab, run Cell 1+2 first. This also tries auto-load)
try:
    config; SERVER_FILES; JAVA
except NameError:
    from pathlib import Path
    import json as _json, shutil as _shutil
    try:
        from google.colab import drive as _drive
        _drive.mount('/content/drive')
    except Exception: pass
    _ROOT = Path('/content/drive/MyDrive/MinecraftServers')
    _cands = [f for f in _ROOT.iterdir() if f.is_dir() and (f/'config.json').exists()] if _ROOT.exists() else []
    if not _cands: raise RuntimeError('No server found. Run Cell 1+2 first with same SERVER_NAME.')
    _pick = [f for f in _cands if f.name=='my-smp']
    SERVER_DIR = (_pick[0] if _pick else _cands[0])
    config = _json.loads((SERVER_DIR/'config.json').read_text())
    SERVER_NAME = config['server_name']
    SERVER_FILES = SERVER_DIR/'server'
    MODS_DIR = SERVER_FILES/'mods'
    PLUGINS_DIR = SERVER_FILES/'plugins'
    BACKUPS_DIR = SERVER_DIR/'backups'
    ROOT = _ROOT
    JAVA = _shutil.which('java')
    print(f"Recovered: {SERVER_NAME} | {config['loader']} {config['minecraft_version']}")

# CELL 8 - Backup anytime
import shutil
from datetime import datetime
send('save-all'); send('save-off')
__import__('time').sleep(3)
z = str(BACKUPS_DIR/(config['server_name']+'_'+datetime.now().strftime('%Y-%m-%d_%H-%M')))
shutil.make_archive(z,'zip',SERVER_FILES)
send('save-on')
print('Backup:', z+'.zip')


[CMD] save-all
[CMD] save-off
Starting net.fabricmc.loader.impl.game.minecraft.BundlerClassPathCapture
[07:08:03] [main/INFO]: Loading Minecraft 1.20.1 with Fabric Loader 0.19.5
[07:08:04] [main/INFO]: Loading 56 mods:
	- bluemap 5.3
	- cloth-config 11.1.136
	   \-- cloth-basic-math 0.6.1
	- fabric-api 0.92.12+1.20.1
	   |-- fabric-api-base 0.4.32+1802ada577
	   |-- fabric-api-lookup-api-v1 1.6.37+1802ada577
	   |-- fabric-biome-api-v1 13.0.15+93aa238b77
	   |-- fabric-block-api-v1 1.0.12+1802ada577
	   |-- fabric-block-view-api-v2 1.0.3+924f046a77
	   |-- fabric-command-api-v1 1.2.35+f71b366f77
	   |-- fabric-command-api-v2 2.2.14+1802ada577
	   |-- fabric-commands-v0 0.2.52+df3654b377
	   |-- fabric-containers-v0 0.1.68+df3654b377
	   |-- fabric-content-registries-v0 4.0.13+1802ada577
	   |-- fabric-convention-tags-v1 1.5.6+1802ada577
	   |-- fabric-crash-report-info-v1 0.2.20+1802ada577
	   |-- fabric-data-attachment-api-v1 1.0.2+de0fd6d177
	   |-- fabric-data-generation-api-v1 12.4